# ChatBot AI Agent with LangChain and OpenAI

### Install

In [23]:
# !python -m pip install openai==1.55.2 # from terminal or jupyter notebook
# !python -m pip install langchain==0.3.9 --user # latest as of Nov 2024
# !pip install langchain-openai==0.2.10
# !pip install -U langchain-pinecone==0.2.0 
# !pip install langchain

In [26]:
# !pip install --upgrade pip
# !pip show langchain-pinecone

In [27]:
!py --version

Python 3.13.5


### Data Load

In [1]:
# https://rajpurkar.github.io/SQuAD-explorer/

# from datasets import load_dataset 

# data = load_dataset('squad', split='train')
# df = data.to_pandas()
# df.drop_duplicates(subset='context', keep='first', inplace=True)
# print(df.shape)
# df.head(2)

from datasets import load_dataset

data = load_dataset('squad', split='train')
df = data.to_pandas()
df.drop_duplicates(subset = 'context', keep='first', inplace=True)
print(df.shape)
df.head()

(18891, 5)


,id,title,context,question,answers
0,5733be284776f41900661182,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",To whom did the Virgin Mary allegedly appear i...,"{'text': ['Saint Bernadette Soubirous'], 'answ..."
5,5733bf84d058e614000b61be,University_of_Notre_Dame,"As at most other universities, Notre Dame's st...",When did the Scholastic Magazine of Notre dame...,"{'text': ['September 1876'], 'answer_start': [..."
10,5733bed24776f41900661188,University_of_Notre_Dame,The university is the major seat of the Congre...,Where is the headquarters of the Congregation ...,"{'text': ['Rome'], 'answer_start': [119]}"
15,5733a6424776f41900660f51,University_of_Notre_Dame,The College of Engineering was established in ...,How many BS level degrees are offered in the C...,"{'text': ['eight'], 'answer_start': [487]}"
20,5733a70c4776f41900660f64,University_of_Notre_Dame,All of Notre Dame's undergraduate students are...,What entity provides help with the management ...,"{'text': ['Learning Resource Center'], 'answer..."


In [2]:
df.columns


Index(['id', 'title', 'context', 'question', 'answers'], dtype='object')

In [3]:
df = data.to_pandas()


In [4]:
df.iloc[0]['context']


'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.'

In [5]:
df.iloc[0]['question']


'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?'

In [6]:
df.iloc[0]['answers']


{'text': array(['Saint Bernadette Soubirous'], dtype=object),
 'answer_start': array([515])}

In [7]:
sum(df['context'].duplicated())

68708

In [8]:
sum(df['question'].duplicated())

244

In [9]:
# sum(df['answers'].duplicated())

In [10]:
df.drop_duplicates(subset='context', keep='first', inplace=True)

In [11]:
df.shape

(18891, 5)

### Embedding API

In [12]:
# This block is older code using openai==0.28. 
# You can still use it but make sure to install : pip install openai==0.28

import openai
# OPENAI_API_KEY = ""
openai.api_key = OPENAI_API_KEY
MODEL  = "text-embedding-ada-002"

# res = openai.Embedding.create(input = "I love openai", engine = MODEL) 
# embedding = response['data'][0]['embedding']
# print(embedding)




In [13]:
# This block uses lates (Nov 2024) version of openai : # openai.__version__  #'1.55.2'
# from openai import OpenAI
# import os

# MODEL = "text-embedding-ada-002"

# client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", OPENAI_API_KEY)) # first one will work if set as env var, second one explicit one

# response = client.embeddings.create(
#     input="I love openai",
#     model=MODEL
# )

# # usage
# embedding  = (response.data)[0].embedding # new code for v-1.55.2
# print(embedding)


from openai import OpenAI

client = OpenAI(
    # api_key=""
)

response = client.embeddings.create(
    model="text-embedding-3-small",
    input="Hello OpenAI"
)

embedding = response.data[0].embedding

print(len(embedding))


1536


In [14]:
from openai import OpenAI

# client = OpenAI(api_key="")

MODEL = "text-embedding-3-small"

def get_embedding(text, model):
    text = text.replace("\n", " ")

    res = client.embeddings.create(
        input=text,
        model=model
    )

    return res.data[0].embedding

vec = get_embedding(
    "I am trying a new text And see what happens",
    MODEL
)

print(len(vec))

1536


In [15]:
# helper function
def get_embedding(text, model):
    text = text.replace("\n", " ")
    res = client.embeddings.create(input = text, model = model) # engine replaced by model in v-1.55.2
    return (response.data)[0].embedding

vec = get_embedding("I am trying a new text \n And see what happens", MODEL)
print(f"required vector dim: {len(vec)}") # dimension of the index"

required vector dim: 1536


### Vector DB Setup

In [16]:
# db of 1536 dimension

from pinecone import Pinecone, ServerlessSpec
# API_KEY = ""
pc = Pinecone(api_key = API_KEY)

# pc.create_index("ai-agent", dimension=1536, metric='dotproduct',
#                      spec=ServerlessSpec(cloud="aws", region="us-east-1"))
index = pc.Index("ai-agent")


In [17]:
# from pinecone import Pinecone

# pc = Pinecone(api_key="")

# pc.delete_index("medium-data")

### Indexing

In [18]:
# df_sample = df.sample(1000, random_state=45)
# batch_size = 100 # free tier limit 20 RPM in 2023 now 3000


df_sample=df.sample(100, random_state=45)
batch_size = 10

In [19]:
from tqdm.auto import tqdm
import time

In [20]:
%%time

for i in tqdm(range(0, len(df_sample), batch_size)):
    i_end = min(i+batch_size, len(df_sample) )
    print(i, i_end)
    batch = df_sample.iloc[i:i_end]
    meta_data=[{"title" :row['title'],"question": row['question'],"answers": row['answers'], "context":row['context']} for i, row in batch.iterrows()]
# print(meta_data)

    docs = batch['context'].tolist()
    emb_vectors = [get_embedding(doc, MODEL) for doc in docs]
    ids = batch['id'].tolist()

    # to_upsert = zip(ids, emb_vectors, meta_data)
    # index.upsert(vectors=to_upsert)
    time.sleep(10)

  0%|          | 0/10 [00:00<?, ?it/s]

0 10
10 20
20 30
30 40
40 50
50 60
60 70
70 80
80 90
90 100
CPU times: total: 547 ms
Wall time: 2min 19s


In [21]:
batch = df_sample.iloc[0:2]

for i, row in batch.iterrows():
    print(i, row, "\n\n")
    print(row['context'], "\n")

38414 id                                   5725d45b89a1e219009abf62
title                                                  Israel
context     Chess is a leading sport in Israel and is enjo...
question                    What's a leading sport in Israel?
answers              {'text': ['Chess'], 'answer_start': [0]}
Name: 38414, dtype: object 


Chess is a leading sport in Israel and is enjoyed by people of all ages. There are many Israeli grandmasters and Israeli chess players have won a number of youth world championships. Israel stages an annual international championship and hosted the World Team Chess Championship in 2005. The Ministry of Education and the World Chess Federation agreed upon a project of teaching chess within Israeli schools, and it has been introduced into the curriculum of some schools. The city of Beersheba has become a national chess center, with the game being taught in the city's kindergartens. Owing partly to Soviet immigration, it is home to the largest numbe

In [22]:
batch = df_sample.iloc[0:10]

for i, row in batch.iterrows():
    print({"title" :row['title'],"question": row['question'],"answers": row['answers'], "context":row['context']},"\n")

{'title': 'Israel', 'question': "What's a leading sport in Israel?", 'answers': {'text': array(['Chess'], dtype=object), 'answer_start': array([0])}, 'context': "Chess is a leading sport in Israel and is enjoyed by people of all ages. There are many Israeli grandmasters and Israeli chess players have won a number of youth world championships. Israel stages an annual international championship and hosted the World Team Chess Championship in 2005. The Ministry of Education and the World Chess Federation agreed upon a project of teaching chess within Israeli schools, and it has been introduced into the curriculum of some schools. The city of Beersheba has become a national chess center, with the game being taught in the city's kindergartens. Owing partly to Soviet immigration, it is home to the largest number of chess grandmasters of any city in the world. The Israeli chess team won the silver medal at the 2008 Chess Olympiad and the bronze, coming in third among 148 teams, at the 2010 Ol

In [23]:
meta_data = [{"title" :row['title'],"question": row['question'],"answers": row['answers'], "context":row['context']} for i, row in batch.iterrows()]
# print(meta_data)
len(meta_data)

10

In [24]:
batch['context']
# len(batch)

38414    Chess is a leading sport in Israel and is enjo...
29102    To the east of the county are upland areas lea...
83084    In 525 BC, the powerful Achaemenid Persians, l...
59863    In March 2013, Governor Rick Snyder declared a...
6411     According to this narrative, shortly after the...
33036    As a uniformed military service, the Army is p...
82747    By the end of the 18th century the population ...
26028    The state has the 12th-largest state economy i...
29332    The U.S. Social Security Administration (SSA),...
73363    In general relativity, a vanishing stress-ener...
Name: context, dtype: object

In [25]:
# embedding function from OpenAI, old code  won't work anymore. instead use langchain-openai as shown in next cell
# from langchain.embeddings.openai import OpenAIEmbeddings

# model_name = "text-embedding-ada-002"

# embed = OpenAIEmbeddings(model = model_name, openai_api_key= OPENAI_API_KEY)

In [26]:
# https://pypi.org/project/langchain-openai/
# pip install langchain-openai==0.2.10

from langchain_openai import OpenAIEmbeddings 

MODEL = "text-embedding-ada-002"
embed = OpenAIEmbeddings(
    model = MODEL,
    openai_api_key= OPENAI_API_KEY)

# Usage:
# doc1 = "Hello how are you"
# doc2 = "Hello everyone!"
# embed.embed_query(doc) # single doc
# embed.embed_documents([doc1, doc2])  # output will be list of list

In [27]:
from tqdm.auto import tqdm
import time

In [28]:
%%time
for i in tqdm(range(0, len(df_sample), batch_size)):
    i_end = min(i+batch_size, len(df_sample))
#     print(i, i_end)
    batch = df_sample.iloc[i:i_end]
    meta_data = [{"titile" : row['title'], 
              "context": row['context']} 
             for i, row in batch.iterrows()]
    
    # embedding  
    docs = batch['context'].tolist()  # pd.Series to python list
#     emb_vectors = [get_embedding(doc, MODEL) for doc in docs] 
    emb_vectors = embed.embed_documents(docs) # list of list

    ids = batch['id'].tolist()
    
    # upsert
    to_upsert = zip(ids, emb_vectors, meta_data)    
    index.upsert(vectors=to_upsert)
    
    # time.sleep(20) # 8s for 50 data points, this was needed when free tier had rate limit to 20RPM, no need anymore

    
# df.shape[0]/3600 # 5 hrs to load , free tier will take 15hrs
# # 14000 records/dollar

  0%|          | 0/10 [00:00<?, ?it/s]

CPU times: total: 1.52 s
Wall time: 17.1 s


### Using

In [32]:
def get_embedding2(text):
    text = text.replace("\n", " ")
    res = client.embeddings.create(input = text, 
                                  model = "text-embedding-ada-002")
    return (res.data)[0].embedding

# get_embedding2("tabula rasa")

In [35]:
# !pip install langchain==0.1.20
# !pip install langchain-openai==0.1.7
# !pip install langchain-pinecone==0.1.1
# !pip install pinecone-client==3.2.2
# !pip install openai==1.30.1

In [36]:
# Updated code with new libraries and classes
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings 

MODEL = "text-embedding-ada-002"

# Initialize the vector store with the correct embedding method
embeddings = OpenAIEmbeddings(model=MODEL, api_key=OPENAI_API_KEY)


vectorstore = PineconeVectorStore(index, embeddings, "context", pinecone_api_key= API_KEY) # df['context'] column is the actual text field to search from

# Perform the similarity search, pure semantic, nothing genrative
query = "destruction of US fifth fleet"
results = vectorstore.similarity_search(query, k=2)


In [37]:
results

[Document(page_content='Acting on State Secretary Acheson\'s recommendation, President Truman ordered General MacArthur to transfer matériel to the Army of the Republic of Korea while giving air cover to the evacuation of U.S. nationals. The President disagreed with advisers who recommended unilateral U.S. bombing of the North Korean forces, and ordered the US Seventh Fleet to protect the Republic of China (Taiwan), whose government asked to fight in Korea. The United States denied ROC\'s request for combat, lest it provoke a communist Chinese retaliation. Because the United States had sent the Seventh Fleet to "neutralize" the Taiwan Strait, Chinese premier Zhou Enlai criticized both the UN and U.S. initiatives as "armed aggression on Chinese territory."', metadata={'titile': 'Korean_War'}),
 Document(page_content="On 7 June 1981, the Israeli air force destroyed Iraq's sole nuclear reactor, in order to impede Iraq's nuclear weapons program. The reactor was under construction just outs

In [ ]:
# embed.embed_query("embedding single document")

# embed.embed_documents(["first doc", "second doc"])

In [ ]:
# !pip install -U langchain langchain-openai openai
# !pip install -U langchain langchain-openai openai
# !pip uninstall -y langchain langchain-core langchain-community langchain-openai pydantic
# !pip install langchain==0.1.20
# !pip install langchain-openai==0.1.7
# !pip install openai==1.30.1
# !pip install pydantic==2.7.4



### Define QA Agent

In [38]:
from langchain_openai import ChatOpenAI
from langchain.chains.conversation.memory import ConversationBufferWindowMemory
from langchain.chains import RetrievalQA

# OpenAI LLM
llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    model='gpt-3.5-turbo',
    temperature=0.0
)

# conversational memory
conv_mem = ConversationBufferWindowMemory(
    memory_key='chat_history',
    k=5,
    return_messages=True
)

# retrieval qa
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)

print("QA Chain Created Successfully")

QA Chain Created Successfully


### Invoking Retrieval QA

In [39]:
query = "Which year university of notredame was established"
qa.invoke(query) # retrieving the info

{'query': 'Which year university of notredame was established',
 'result': 'The University of Notre Dame was established in 1842.'}

In [41]:
query = "who established the university of notredame"
qa.invoke(query)

{'query': 'who established the university of notredame',
 'result': 'The University of Notre Dame was established by Father Edward Sorin, who was a French priest and the founder of the Congregation of Holy Cross. Notre Dame was founded in 1842 in Notre Dame, Indiana.'}

In [43]:
from langchain.agents import Tool

tools = [
    Tool(
    name = 'Knowledge Base',
    func = qa.invoke,
    description = ('use this when answering based on knwowledge')
    )
]

In [44]:
from langchain.agents import initialize_agent
from langchain.agents import AgentType

agent = initialize_agent(
    agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
    tools=tools,
    llm=llm,
    verbose=True,
    max_iterations=3,
    early_stopping_method='generate',
    memory=conv_mem 
)

C:\Users\tharu\anaconda3\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 0.3.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  warn_deprecated(


In [45]:
agent("when was university of notredame established") # chat gpt kind

C:\Users\tharu\anaconda3\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(




> Entering new AgentExecutor chain...
```json
{
    "action": "Knowledge Base",
    "action_input": "University of Notre Dame establishment date"
}
```
Observation: {'query': 'University of Notre Dame establishment date', 'result': 'The University of Notre Dame was established on November 26, 1842.'}
Thought:```json
{
    "action": "Final Answer",
    "action_input": "The University of Notre Dame was established on November 26, 1842."
}
```

> Finished chain.


{'input': 'when was university of notredame established',
 'chat_history': [],
 'output': 'The University of Notre Dame was established on November 26, 1842.'}

In [46]:
agent("who founded the university")



> Entering new AgentExecutor chain...
```json
{
    "action": "Knowledge Base",
    "action_input": "University of Notre Dame founding"
}
```
Observation: {'query': 'University of Notre Dame founding', 'result': 'The University of Notre Dame was founded on November 26, 1842, by Rev. Edward Sorin, a priest of the Congregation of Holy Cross. The university is located in Notre Dame, Indiana, near South Bend. Notre Dame is known for its strong Catholic identity and academic excellence.'}
Thought:```json
{
    "action": "Final Answer",
    "action_input": "The University of Notre Dame was founded on November 26, 1842, by Rev. Edward Sorin."
}
```

> Finished chain.


{'input': 'who founded the university',
 'chat_history': [HumanMessage(content='when was university of notredame established'),
  AIMessage(content='The University of Notre Dame was established on November 26, 1842.')],
 'output': 'The University of Notre Dame was founded on November 26, 1842, by Rev. Edward Sorin.'}

In [47]:
agent("20+6")



> Entering new AgentExecutor chain...
```json
{
    "action": "Final Answer",
    "action_input": "26"
}
```

> Finished chain.


{'input': '20+6',
 'chat_history': [HumanMessage(content='when was university of notredame established'),
  AIMessage(content='The University of Notre Dame was established on November 26, 1842.'),
  HumanMessage(content='who founded the university'),
  AIMessage(content='The University of Notre Dame was founded on November 26, 1842, by Rev. Edward Sorin.')],
 'output': '26'}

#### Note on the Rate Limit

Rate Limit: https://github.com/openai/openai-cookbook/blob/main/examples/How_to_handle_rate_limits.ipynb

Retry Options: https://github.com/openai/openai-cookbook/blob/main/examples/How_to_handle_rate_limits.ipynb

### Further Reading

https://arxiv.org/abs/2005.11401 

https://platform.openai.com/docs/models/gpt-3-5